# 04 — 凍結済み研究結果

処理は正式freezeと集計CSVの表示・整合性確認だけです。元値をNotebookに手入力で再作成しません。単位・母数・Pathの区別を読み取ります。

**実行方法**：`notebooks/README.md`の環境を用意し、上からRun All。作業ディレクトリはrepo rootまたはnotebooks。第三者の住所・大容量入力・SUMOは不要です。途中の条件を編集したら、そのセル以降を再実行してください。正式実験条件・freezeは変更しません。

**前提・範囲**：Python 3.12系で確認する小型教材／保存結果表示です。必要な配送を省略した電力量減を省エネ成果としません。量子実機・量子優位性・全需要充足・全システムエネルギーの実証ではありません。Colab実行は未検証です。


In [1]:
from pathlib import Path
import sys, json, math
from decimal import Decimal
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'notebooks/adapters.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('リポジトリrootまたはnotebooksを作業ディレクトリにしてください')
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / 'notebooks'))
import adapters as api
S = '01_research_design/scenario_evaluation/'
M = '01_research_design/methodology/'
Q = '05_src/traffic_simulation/'
L = '05_src/literature_analysis/'


In [2]:
F=S+'final_quantitative_study_20261008/'
method_battery, scale_analysis = api.analysis_api()
f=method_battery.freeze()
display(pd.DataFrame([dict(path='R0',**f['R0']),dict(path='R1',**f['R1'])]))
assert f['R0']['route_available']==78
assert f['R1']['IDENTICAL']+f['R1']['DIFFERENT']==f['R1']['route_available']==64
assert f['R1']['route_available']+f['R1']['unavailable']==78
print('共通block:',f['common_matched']['included_blocks'])

,path,full_science_completed,population,route_available,IDENTICAL,DIFFERENT,identical_trajectories,method_battery_all_conditions,scope,unavailable,trajectory_and_Battery_evaluated
0,R0,78,78,78,78,0,78.0,EXACT_ZERO,FULL_78,NaN,NaN
1,R1,78,78,64,59,5,NaN,NaN,CONDITIONAL_64_OF_78,14.0,64.0


共通block: 64


In [3]:
battery=method_battery.battery_conditions()
display(pd.DataFrame(battery['stage_inputs']))
summary=method_battery.summary()
assert len(summary)==4 and summary['n'].eq(64).all()
assert summary['units'].eq('percentage_points').all()
display(summary[['rolling_case','comparison','n','units','mean','median','positive_count','zero_count','negative_count']])
rolling_case='CRR_LOWER_BOUND'  # 表示条件だけ変更
display(summary[summary.rolling_case==rolling_case])

,stage_id,e_pack_Wh_kg,E_pack_fixed_Wh,E_vehicle_usable_assumed_Wh,m_pack_kg,vehicle_mass_kg,running_mass_kg
0,B0,180.0,20000,18000,111.111111,1100.000000,1155.000000
1,B1,220.0,20000,18000,90.909091,1079.797980,1134.797980
2,B2,250.0,20000,18000,80.000000,1068.888889,1123.888889


,rolling_case,comparison,n,units,mean,median,positive_count,zero_count,negative_count
0,CRR_LOWER_BOUND,B0_TO_B1,64,percentage_points,-0.000248,0.0,1,59,4
1,CRR_LOWER_BOUND,B0_TO_B2,64,percentage_points,-0.000382,0.0,1,59,4
2,CRR_UPPER_BOUND,B0_TO_B1,64,percentage_points,-0.000219,0.0,1,59,4
3,CRR_UPPER_BOUND,B0_TO_B2,64,percentage_points,-0.000338,0.0,1,59,4


,rolling_case,comparison,units,definition,n,mean_decimal,mean,median,min,max,standard_deviation,standard_deviation_ddof,mean_absolute_value,median_absolute_value,positive_count,zero_count,negative_count,source
0,CRR_LOWER_BOUND,B0_TO_B1,percentage_points,T=(Classical-Quantum)_R1-(Classical-Quantum)_R0,64,-0.000248,-0.000248,0.0,-0.010014,0.000485,0.001322,1,0.000263,0.0,1,59,4,01_research_design/scenario_evaluation/matched...
1,CRR_LOWER_BOUND,B0_TO_B2,percentage_points,T=(Classical-Quantum)_R1-(Classical-Quantum)_R0,64,-0.000382,-0.000382,0.0,-0.015425,0.000744,0.002037,1,0.000406,0.0,1,59,4,01_research_design/scenario_evaluation/matched...


In [4]:
validation=scale_analysis.trajectory_validation()
print('保存記録：古典6規模のnative starts after due =',validation['native_starts_after_due_diagnostic'])
print('保存記録：完全なVRPTW native revalidation =',validation['VRPTW_native_revalidation'])
assert validation['native_starts_after_due_diagnostic']==136
assert validation['VRPTW_native_revalidation'] is False

保存記録：古典6規模のnative starts after due = 136
保存記録：完全なVRPTW native revalidation = False


## 指標と解釈
D=ClassicalのBattery削減率−QuantumのBattery削減率、T=D_R1−D_R0、単位ppです。絶対Wh差や正式主仮説の応答差と同一視しません。共通64はR1取得例への条件付き比較で、未取得14を0Whで補いません。

L1静的制約、L2走行完了、L3実時刻時間窓、L4実業務充足は別です。ここで表示した遅延136は古典6規模の保存診断で、R1 Quantumの64例の統計ではありません。全実需充足と総エネルギーのtrade-off改善は未実証です。存在/hash/母数確認は全モデル妥当性検証ではありません。